In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF")
print(f"First Page Preview:{pages[0].page_content[:500]}")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# we are chunking the pdf document we loaded
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100,
    separators=["\n\n","\n","."," "]  # separate at paragraph or line or sentence or word
)

chunks = splitter.split_documents(pages)

len(chunks)

In [ ]:
chunks[0].page_content

In [ ]:
# lets store these into vector db
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embedding=embeddings)

print(f"Vector store created with {vector_store._collection.count()} vectors")

In [ ]:
# Retrieve relevant documents from the vector store
retriever = vector_store.as_retriever(search_kwargs={"k":3})

test_query = "what is VOLTE and how does it improve call quality?"
retrived = retriever.invoke(test_query)

for i,doc in enumerate(retrived,1):
    print(f"----------Chunk {i}----------")
    print(doc.page_content[:300])
    print()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# once the chunks are rettrieved we join the chunks and pass it to the llm for answering the question
def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

system_prompt = """You are a helpful assistant that answers questions based on the context provided.
If the context does not contain enough information , say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

llm = ChatGroq(model = "qwen/qwen3.6-27b",temperature = 0, reasoning_format="parsed", max_tokens = 800)

chain = (
    {"context":retriever |format_docs,"question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

print("Rag chain assembled successfully")


In [ ]:
question = "How does international roaming work and what charges should I expect?"
print(f"question: {question}")
print("Answer:", chain.invoke(question))